# Rank Ordering & Model Comparison

Companion notebook for [Rank Ordering & Model Comparison](https://ml-viz-ruby.vercel.app/courses/model-evaluation/08-rank-ordering-and-model-comparison).

**The problem.** Every metric so far has needed a threshold you have not chosen yet, and has come as a point estimate with no error bar. This notebook builds the two tools that fix that: **concordance** measures ranking quality with no cutoff at all, and **paired tests** say whether a gap between two models survives resampling.

**The arc.** One synthetic credit book, scored by two models, carried through the whole validation pack: c-statistic and Gini, the decile table with lift, gains and KS, then DeLong and McNemar on the two models.

*Colab tip: File → Save a copy in Drive before editing.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

plt.style.use('dark_background')
rng = np.random.default_rng(42)
pd.set_option('display.width', 140)

## 0 — A book to validate

Twelve thousand accounts, five features, a 6% bad rate. Two candidate models: a **full** model on all five features and a **thin** model on three of them. The thin model is deliberately the weaker one, which is what makes the comparison worth testing.

In [ ]:
N, N_FEATURES = 12_000, 5
X = rng.normal(size=(N, N_FEATURES))
beta = np.array([1.1, -0.8, 0.6, 0.4, 0.25])
logit = X @ beta - 3.1                      # intercept tuned for a ~6% bad rate
y = rng.binomial(1, 1 / (1 + np.exp(-logit)))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4,
                                          stratify=y, random_state=0)
full = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
thin = LogisticRegression(max_iter=1000).fit(X_tr[:, :3], y_tr)

s_full = full.predict_proba(X_te)[:, 1]
s_thin = thin.predict_proba(X_te[:, :3])[:, 1]

print(f'{N:,} accounts, bad rate {y.mean():.2%}; test set {len(y_te):,} '
      f'with {y_te.sum()} bads')
print(f'full model AUC: {roc_auc_score(y_te, s_full):.4f}')
print(f'thin model AUC: {roc_auc_score(y_te, s_thin):.4f}')

## 1 — Concordance from scratch

$c$ is a pair count: over every (bad, good) pair, how often is the bad scored higher? Ties count as half a win.

$$c = \frac{C + T/2}{C + D + T}, \qquad D_{xy} = \frac{C - D}{C + D + T} = 2c - 1$$

In [ ]:
def rank_stats(score, label):
    """c-statistic, Somers' D and gamma, straight from the pair counts."""
    pos = score[label == 1][:, None]
    neg = score[label == 0][None, :]
    C = int((pos > neg).sum())
    D = int((pos < neg).sum())
    T = int((pos == neg).sum())
    total = C + D + T
    return {'C': C, 'D': D, 'T': T,
            'c': (C + 0.5 * T) / total,
            'somers_d': (C - D) / total,
            'gini': 2 * ((C + 0.5 * T) / total) - 1,
            'gamma': (C - D) / (C + D)}

st = rank_stats(s_full, y_te)
print({k: (v if isinstance(v, int) else round(v, 4)) for k, v in st.items()})
assert np.allclose(st['c'], roc_auc_score(y_te, s_full)), 'pair count must equal AUC'
print(f"\nc == roc_auc_score  : {st['c']:.6f}")
print(f"Gini == Somers' D   : {st['gini']:.6f} == {st['somers_d']:.6f}")

### What a risk grade costs

Round the score into ten grades, the way a production scorecard does, and recount. Within-grade pairs become ties.

In [ ]:
grades = np.floor(stats.rankdata(s_full) / (len(s_full) + 1) * 10)
coarse = rank_stats(grades, y_te)

print(f"{'':>10} {'raw score':>11} {'10 grades':>11}")
print(f"{'ties':>10} {st['T']:>11,} {coarse['T']:>11,}")
for k in ('c', 'somers_d', 'gamma'):
    print(f'{k:>10} {st[k]:>11.4f} {coarse[k]:>11.4f}')
print('\nc and D fall (information lost); gamma rises (it deletes the ties)')

## 2 — The decile table

Sort by score, cut into ten equal-count bins, and read where the bads landed.

In [ ]:
def decile_table(score, bad, k=10):
    order = np.argsort(-score)
    b = bad[order]
    rows = [{'bin': i, 'n': len(idx), 'bads': int(b[idx].sum())}
            for i, idx in enumerate(np.array_split(np.arange(len(b)), k), 1)]
    t = pd.DataFrame(rows)
    t['goods']    = t['n'] - t['bads']
    t['bad_rate'] = t['bads'] / t['n']
    t['lift']     = t['bad_rate'] / bad.mean()
    t['cum_bad']  = t['bads'].cumsum() / t['bads'].sum()
    t['cum_good'] = t['goods'].cumsum() / t['goods'].sum()
    t['ks']       = t['cum_bad'] - t['cum_good']
    return t

table = decile_table(s_full, y_te)
print(table.round(4).to_string(index=False))
print(f"\ntop-decile lift {table.loc[0, 'lift']:.2f}x, "
      f"capture at 20% {table.loc[1, 'cum_bad']:.1%}, "
      f"KS {table['ks'].max():.4f} at bin {int(table['ks'].idxmax()) + 1}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
for s, name, col in [(s_full, 'full', '#14b8a6'), (s_thin, 'thin', '#6366f1')]:
    fpr, tpr, _ = roc_curve(y_te, s)
    ax.plot(fpr, tpr, color=col, label=f'{name} (AUC {roc_auc_score(y_te, s):.3f})')
ax.plot([0, 1], [0, 1], '--', color='#475569')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC'); ax.legend()

ax = axes[1]
pop = np.r_[0, table['bin'] / 10]
ax.plot(pop, np.r_[0, table['cum_bad']], 'o-', color='#14b8a6', label='bads')
ax.plot(pop, np.r_[0, table['cum_good']], 'o-', color='#6366f1', label='goods')
ax.plot([0, 1], [0, 1], '--', color='#475569')
kb = int(table['ks'].idxmax())
ax.vlines((kb + 1) / 10, table.loc[kb, 'cum_good'], table.loc[kb, 'cum_bad'],
          color='#eab308', lw=2.5, label=f"KS {table['ks'].max():.3f}")
ax.set_xlabel('% targeted'); ax.set_ylabel('% captured')
ax.set_title('Cumulative gains'); ax.legend(loc='lower right')

ax = axes[2]
ax.bar(table['bin'], table['lift'],
       color=['#14b8a6' if l >= 1 else '#6366f1' for l in table['lift']])
ax.axhline(1, ls='--', color='#94a3b8')
ax.set_xlabel('decile (1 = riskiest)'); ax.set_ylabel('lift'); ax.set_title('Lift')
plt.tight_layout(); plt.show()

**What to notice.** The three panels answer three different questions from the same predictions: the ROC says how well the model ranks, the gains curve says what you capture at a given budget, and the lift bars say whether the ordering is monotone. Only the first is prevalence-free.

## 3 — Is the full model really better?

Both models scored the **same** test accounts, so their AUC estimates are correlated and the difference has a covariance term:

$$\operatorname{var}(\hat{A} - \hat{B}) = \operatorname{var}(\hat{A}) + \operatorname{var}(\hat{B}) - 2\operatorname{cov}(\hat{A}, \hat{B})$$

In [ ]:
def delong_test(scores, label):
    """DeLong's test for two correlated AUCs, via placement values."""
    m, n = int(label.sum()), int((1 - label).sum())
    V10, V01, aucs = [], [], []
    for s in scores:
        pos, neg = s[label == 1][:, None], s[label == 0][None, :]
        psi = (pos > neg) + 0.5 * (pos == neg)
        V10.append(psi.mean(axis=1)); V01.append(psi.mean(axis=0))
        aucs.append(psi.mean())
    S = np.cov(np.array(V10), ddof=1) / m + np.cov(np.array(V01), ddof=1) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return dict(aucs=np.array(aucs), S=S, se=np.sqrt(var), z=z,
                p=2 * stats.norm.sf(abs(z)))

d = delong_test([s_full, s_thin], y_te)
gap = d['aucs'][0] - d['aucs'][1]
corr = d['S'][0, 1] / np.sqrt(d['S'][0, 0] * d['S'][1, 1])
print(f"AUCs {d['aucs'][0]:.4f} vs {d['aucs'][1]:.4f}, gap {gap:.4f}")
print(f"corr(AUC_full, AUC_thin) = {corr:.3f}")
print(f"paired  : se {d['se']:.4f}, z {d['z']:.2f}, p {d['p']:.2e}")

se_naive = np.sqrt(d['S'][0, 0] + d['S'][1, 1])
z_naive = gap / se_naive
print(f"unpaired: se {se_naive:.4f}, z {z_naive:.2f}, "
      f"p {2 * stats.norm.sf(abs(z_naive)):.4f}")
print(f"\n95% CI for the gap: [{gap - 1.96 * d['se']:.4f}, {gap + 1.96 * d['se']:.4f}]")

The unpaired standard error is several times the paired one on identical data. On a gap this large both tests agree, but on the 0.01-to-0.02 gaps that real model iterations produce, that factor is the difference between a finding and a shrug.

In [ ]:
# McNemar at a fixed threshold: only the disagreements matter.
thr = np.quantile(s_full, 0.90)             # decline the riskiest 10%
pred_full = (s_full >= thr).astype(int)
pred_thin = (s_thin >= np.quantile(s_thin, 0.90)).astype(int)

ok_f, ok_t = pred_full == y_te, pred_thin == y_te
a = int((ok_f & ok_t).sum());  b = int((ok_f & ~ok_t).sum())
c = int((~ok_f & ok_t).sum()); dd = int((~ok_f & ~ok_t).sum())
print(f'both right {a}, full only {b}, thin only {c}, both wrong {dd}')

chi2 = (abs(b - c) - 1) ** 2 / (b + c)
print(f'\nMcNemar chi2 = {chi2:.3f}, p = {stats.chi2.sf(chi2, 1):.2e}')
print(f'exact binomial p = {2 * stats.binom.cdf(min(b, c), b + c, 0.5):.2e}')
print(f'accuracy {ok_f.mean():.4f} vs {ok_t.mean():.4f}, '
      f'decided on {b + c} disagreements out of {len(y_te)}')

**What to notice: the two tests disagree, and both are right.** DeLong says the full model ranks significantly better (p = 0.0008). McNemar at the 90th-percentile cutoff cannot resolve a difference in accuracy (p = 0.18), because at that one threshold the two models disagree on only 248 of 4,800 accounts and the split is close to even. Better ranking does not automatically mean better decisions at the threshold you happen to have chosen, which is why a validation pack reports both.

## 4 — Tradeoffs: which number for which question

| Question | Metric | Prevalence-dependent? |
|---|---|---|
| How well does it rank? | c-statistic / Gini | no |
| How separated are the score distributions? | KS | barely |
| What do I catch at a 10% review budget? | gains / capture | yes |
| How much better than random targeting? | lift | **yes**, capped at $1/\pi$ |
| Are the predicted probabilities right? | calibration curve, ECE | yes |
| Is model A genuinely better than B? | paired test with an interval | n/a |

**Failure modes**: concordance on binned scores measures your binning; lift compared across base rates compares populations; unpaired tests on a shared test set hide real improvements; a test set reused across 200 experiments has no p-values left to give.

In [ ]:
# One table, one sentence: the validation summary from the lesson.
lo, hi = gap - 1.96 * d['se'], gap + 1.96 * d['se']
print(f'On {len(y_te):,} held-out accounts the full model scores '
      f"c = {d['aucs'][0]:.3f} against the thin model's {d['aucs'][1]:.3f}, "
      f'a paired DeLong difference of {gap:.3f} '
      f"(95% CI {lo:.3f} to {hi:.3f}, p = {d['p']:.1e}), and captures "
      f"{table.loc[1, 'cum_bad']:.0%} of bads in the top two deciles.")

## 5 — ✏️ Your turn

### Exercise 1: Gini and capture from a decile table

From per-decile bad counts alone, compute the top-decile lift, the capture rate at 30%, and the KS.

In [ ]:
bads = np.array([185, 110, 70, 45, 30, 22, 15, 10, 8, 5])
n_bin = 1000

def exercise_1(bads, n_bin):
    goods = None        # TODO(you): accounts per bin minus bads
    lift1 = None        # TODO(you): decile 1 bad rate / overall bad rate
    capture30 = None    # TODO(you): share of all bads in the top three deciles
    ks = None           # TODO(you): max over bins of cum_bad% - cum_good%
    return lift1, capture30, ks

In [ ]:
l1, c30, ks = exercise_1(bads, n_bin)
assert abs(l1 - 3.70) < 1e-9, f'top-decile lift should be 3.70, got {l1}'
assert abs(c30 - 0.73) < 1e-9, f'capture at 30% should be 0.73, got {c30}'
assert abs(ks - 0.4526315789) < 1e-9, f'KS should be 0.4526, got {ks}'
print(f'lift {l1:.2f}x, capture@30% {c30:.0%}, KS {ks:.4f}')

<details>
<summary>Solution</summary>

```python
def exercise_1(bads, n_bin):
    goods = n_bin - bads
    base = bads.sum() / (n_bin * len(bads))
    lift1 = (bads[0] / n_bin) / base                 # 0.185 / 0.05 = 3.70
    capture30 = bads[:3].sum() / bads.sum()          # 365 / 500 = 0.73
    ks = np.max(np.cumsum(bads) / bads.sum()
                - np.cumsum(goods) / goods.sum())    # 0.4526, at decile 3
    return lift1, capture30, ks
```

The KS peaks at decile 3, where 73.0% of bads sit above the cut against 27.7% of goods.
</details>

### Exercise 2: the variance of an AUC difference

Given the DeLong structural matrix $S$ for two models, return the paired standard error, the unpaired one, and both z statistics. The point of the exercise is to see the ratio.

In [ ]:
def exercise_2(S, gap):
    se_paired = None    # TODO(you): sqrt(S00 + S11 - 2*S01)
    se_naive = None     # TODO(you): sqrt(S00 + S11)
    z_paired = None     # TODO(you)
    z_naive = None      # TODO(you)
    return se_paired, se_naive, z_paired, z_naive

In [ ]:
S_demo = np.array([[6.70e-5, 4.84e-5], [4.84e-5, 6.70e-5]])
sp, sn, zp, zn = exercise_2(S_demo, 0.021)
assert abs(sp - 0.0060992) < 1e-6, f'paired se should be 0.006099, got {sp}'
assert abs(sn - 0.0115758) < 1e-6, f'unpaired se should be 0.011576, got {sn}'
assert abs(zp - 3.4431) < 1e-3 and abs(zn - 1.8141) < 1e-3, 'check the z values'
print(f'paired  : se {sp:.5f}, z {zp:.2f}, p {2 * stats.norm.sf(zp):.4f}')
print(f'unpaired: se {sn:.5f}, z {zn:.2f}, p {2 * stats.norm.sf(zn):.4f}')
print('same 0.021 gap, opposite verdicts at alpha = 0.05')

<details>
<summary>Solution</summary>

```python
def exercise_2(S, gap):
    se_paired = np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    se_naive = np.sqrt(S[0, 0] + S[1, 1])
    return se_paired, se_naive, gap / se_paired, gap / se_naive
```

The two AUC estimates correlate at 0.72 here, which halves the standard error and moves the p-value from 0.070 to 0.0006. Nothing about the data changed.
</details>

## 6 — Key takeaways

- **$c = (C + T/2)/(C+D+T)$ is the AUC**, and Gini = Somers' D = $2c - 1$. One pair count, three names.
- **Binning the score raises ties**, which lowers $c$ and Somers' D and *raises* gamma. Compute on raw scores.
- **The decile table answers the operational question** an AUC cannot: lift, capture and KS, with lift alone depending on the base rate.
- **Model comparison is paired.** DeLong for AUCs, McNemar at a threshold, resampling for anything else.
- **Report the interval.** "0.021, CI 0.009 to 0.033" is a decision; "p < 0.05" is not.

**Go deeper:** [concordance and rank statistics](https://ml-viz-ruby.vercel.app/wiki/concordance-and-rank-statistics), [decile analysis](https://ml-viz-ruby.vercel.app/wiki/decile-analysis-and-lift), [comparing model performance](https://ml-viz-ruby.vercel.app/wiki/comparing-model-performance).